# MyMarket: train the impact model (manual step M6)

Food staples only: rice, egg, cooking oil, chicken.

Before you start:
1. Build `examples.parquet` locally with `python scripts/05_build_examples.py`.
2. Upload it to a Google Drive folder named `mymarket`.
3. Runtime, Change runtime type, T4 GPU, Save. Then Connect.

Read the label counts in cell 2 before training. If a product has under 300
examples the result for that product means very little.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q sentence-transformers lightgbm scikit-learn pyarrow

## 1. Load the examples

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd

P = '/content/drive/MyDrive/mymarket'
df = pd.read_parquet(f'{P}/examples.parquet')
df = df[df['split'].isin(['train', 'val', 'test'])].reset_index(drop=True)

print(df.shape)
print(df['label'].value_counts())
print(df.groupby(['product', 'split']).size().unstack(fill_value=0))

assert set(df['split']) == {'train', 'val', 'test'}, df['split'].unique()

## 2. Check the purge gap before anything else

If the gap is smaller than the horizon the split leaks and every number
below is meaningless.

In [ ]:
horizon = int(df['horizon_days'].iloc[0])
train_end = df.loc[df.split == 'train', 'date'].max()
val_start = df.loc[df.split == 'val', 'date'].min()
val_end = df.loc[df.split == 'val', 'date'].max()
test_start = df.loc[df.split == 'test', 'date'].min()

print('horizon', horizon)
print('train ends', train_end.date(), 'val starts', val_start.date(),
      'gap', (val_start - train_end).days)
print('val ends', val_end.date(), 'test starts', test_start.date(),
      'gap', (test_start - val_end).days)
assert (val_start - train_end).days >= horizon
assert (test_start - val_end).days >= horizon

## 3. Embed the text

Saved to Drive, so a disconnect does not cost you the slow step.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device', dev)

model = SentenceTransformer('intfloat/multilingual-e5-small', device=dev)
texts = ['query: ' + t for t in df['text'].fillna('').tolist()]
emb = model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
np.save(f'{P}/emb.npy', emb)
print(emb.shape)

# After a disconnect, run this instead of re-embedding:
# emb = np.load(f'{P}/emb.npy')

## 4. Features: embeddings, product one hot, event flags

The column order must match `src/mymarket/model/features.py` or the saved
model will not work in the server.

In [ ]:
import scipy.sparse as sp
from sklearn.preprocessing import OneHotEncoder

event_cols = [c for c in df.columns if c.startswith('ev_')]
print(len(event_cols), 'event columns')

ohe = OneHotEncoder(handle_unknown='ignore')
Xp = ohe.fit_transform(df[['product']])
Xe = df[event_cols].fillna(0).to_numpy(dtype='float32')
X = sp.hstack([sp.csr_matrix(emb), sp.csr_matrix(Xp), sp.csr_matrix(Xe)]).tocsr()

y = df['label'].to_numpy()
tr = (df['split'] == 'train').to_numpy()
va = (df['split'] == 'val').to_numpy()
te = (df['split'] == 'test').to_numpy()
print(X.shape, tr.sum(), va.sum(), te.sum())

## 5. Train and compare on validation

In [ ]:
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

lr = LogisticRegression(max_iter=3000, class_weight='balanced', C=1.0)
lr.fit(X[tr], y[tr])
lr_score = f1_score(y[va], lr.predict(X[va]), average='macro', zero_division=0)
print('logreg val macroF1', round(lr_score, 4))

gbm = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05,
                         class_weight='balanced', verbose=-1)
gbm.fit(X[tr], y[tr])
gbm_score = f1_score(y[va], gbm.predict(X[va]), average='macro', zero_division=0)
print('lightgbm val macroF1', round(gbm_score, 4))

maj = pd.Series(y[tr]).mode()[0]
print('majority val macroF1',
      round(f1_score(y[va], [maj] * va.sum(), average='macro', zero_division=0), 4))

best, best_name = (gbm, 'lightgbm') if gbm_score > lr_score else (lr, 'logreg')
print('picked', best_name)

## 6. Calibrate on validation, then touch the test set once

Calibration is skipped when validation is missing a class, because sklearn
fails deep inside its cross validation in that case. An uncalibrated model
still predicts, its probabilities are just less trustworthy.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, confusion_matrix

known, present = set(best.classes_), set(np.unique(y[va]))
missing = known - present

if missing or va.sum() < 30:
    cal, calibration = best, f'skipped, missing {sorted(missing)} or too few rows'
else:
    folds = max(2, min(5, min(int((y[va] == c).sum()) for c in present)))
    try:
        from sklearn.frozen import FrozenEstimator
        cal = CalibratedClassifierCV(FrozenEstimator(best), method='sigmoid', cv=folds)
    except ImportError:
        cal = CalibratedClassifierCV(best, method='sigmoid', cv='prefit')
    cal.fit(X[va], y[va])
    calibration = f'sigmoid, cv={folds}'
print('calibration:', calibration)

pred = cal.predict(X[te])
proba = cal.predict_proba(X[te])
print(classification_report(y[te], pred, zero_division=0))
print(confusion_matrix(y[te], pred, labels=cal.classes_), list(cal.classes_))

test_f1 = f1_score(y[te], pred, average='macro', zero_division=0)
maj_f1 = f1_score(y[te], [maj] * te.sum(), average='macro', zero_division=0)
print('TEST macroF1', round(test_f1, 4))
print('majority TEST macroF1', round(maj_f1, 4))

## 7. Per product breakdown

In [ ]:
te_df = df[te].copy()
te_df['pred'] = pred
for product, g in te_df.groupby('product'):
    print(product, 'macroF1',
          round(f1_score(g['label'], g['pred'], average='macro', zero_division=0), 3),
          'n', len(g))

## 8. Confidence interval and alert precision

In [ ]:
rng = np.random.default_rng(0)
yt, pt = y[te], pred
scores = []
for _ in range(1000):
    i = rng.integers(0, len(yt), len(yt))
    scores.append(f1_score(yt[i], pt[i], average='macro', zero_division=0))
lo, hi = np.percentile(scores, [2.5, 97.5])
print('macroF1 95 pct CI', round(lo, 4), round(hi, 4))

conf = proba.max(axis=1)
mask = conf >= 0.7
alert_prec = float((pt[mask] == yt[mask]).mean()) if mask.sum() else None
print('alerts at p>=0.7:', int(mask.sum()), 'precision:', alert_prec)

## 9. Save the four files

Then download them from Drive into `models/` in the project and run
`python scripts/07_evaluate.py` for the rolling origin backtest.

In [ ]:
joblib.dump({
    'model': cal,
    'ohe': ohe,
    'embedder': 'intfloat/multilingual-e5-small',
    'prefix': 'query: ',
    'version': f'v1-{best_name}-colab',
}, f'{P}/model.joblib')

json.dump({str(i): c for i, c in enumerate(cal.classes_)},
          open(f'{P}/label_map.json', 'w'))
json.dump({'min_confidence': 0.55, 'alert_confidence': 0.7},
          open(f'{P}/thresholds.json', 'w'), indent=2)
json.dump({
    'generated_at': pd.Timestamp.utcnow().isoformat(),
    'model': f'v1-{best_name}-colab',
    'calibration': calibration,
    'test_macro_f1': float(test_f1),
    'macro_f1_ci95': [float(lo), float(hi)],
    'baselines_test': {'majority': float(maj_f1)},
    'beats_all_baselines': bool(test_f1 > maj_f1),
    'alert_precision_p70': alert_prec,
    'n_alerts_p70': int(mask.sum()),
    'n_test': int(te.sum()),
    'horizon_days': horizon,
    'note': 'Momentum and keyword baselines are added by scripts/07_evaluate.py.',
}, open(f'{P}/metrics.json', 'w'), indent=2)

print('saved the four files to', P)

## How to read your result

- Above majority and above momentum: good. Report it with the interval.
- About equal to the baselines: the normal outcome for this task. Say so in
  the demo and show the keyword triggers and the price context as the value.
- Below majority: the model is not useful. Try a 28 day horizon, a smaller
  `k` (0.3), or merging products.
- Above 0.9: suspicious. Check the purge gap in cell 2 for leakage.